In [1]:
import numpy as np
from scipy.optimize import least_squares
from scipy.special import gamma
from dataclasses import dataclass
import pandas as pd

In [2]:
country=pd.read_csv(r'E:\zotero\部分论文复刻\JIE2019资本积累和动态效应检验\自我复刻代码\CSV_数据处理与校准输出\01_baseline_country_table.csv')
trade=pd.read_csv(r'E:\zotero\部分论文复刻\JIE2019资本积累和动态效应检验\自我复刻代码\CSV_数据处理与校准输出\02_baseline_trade_table.csv')
parameter=pd.read_csv(r'E:\zotero\部分论文复刻\JIE2019资本积累和动态效应检验\自我复刻代码\CSV_数据处理与校准输出\03_baseline_parameter_table.csv')
country=country.merge(parameter, on='country')

### 静态模型

In [3]:
# 全局标量参数
@dataclass(frozen=True)
class ModelParams:
    theta: float = 4.0
    eta: float = 2.0
    beta: float = 0.96
    sigma: float = 0.5
    delta: float = 0.06
    lamb: float = 0.76
    alpha: float = 0.33

    @property
    def chi(self):
        return self.delta ** (1 - self.lamb)

    @property
    def gamma_term(self):
        return gamma(1 + (1 / self.theta) * (1 - self.eta)) ** (1 / (1 - self.eta))
        

    @property
    def Alpha(self):
        return self.alpha ** (-self.alpha) * (1 - self.alpha) ** (self.alpha - 1)
    
    @property
    def phi_state_1(self):
        return 1/self.lamb
    
    @property
    def phi_state_2(self):
        return self.delta-1/self.lamb

    @property
    def q(self):
        return 1/self.beta-1
p: ModelParams = ModelParams()

In [4]:
@dataclass(frozen=True)
class CountryParams:
    nu_m: np.ndarray
    nu_c: np.ndarray
    nu_x: np.ndarray
    T_m: np.ndarray
    A_c: np.ndarray
    A_x: np.ndarray
    F: np.ndarray
    L: np.ndarray
    K: np.ndarray
    @property
    def N(self):
        return len(self.nu_m)

cp: CountryParams = CountryParams(
    nu_m=country["nu_m0"].values,
    nu_c=country["nu_c0"].values,
    nu_x=country["nu_x0"].values,
    T_m=country["T_m0"].values,
    A_c=country["A_c0"].values,
    A_x=country["A_x0"].values,
    F=country["F"].values,
    L=country["L0"].values,
    K=country["K0"].values,
)

In [5]:
#这是第一版写法，这是不对的，在数值中，特别是多国模型中，能用数组就用数组，因此MATLAB还是非常方便推荐的
# def u_j(r_j,w_j,P_mj,p,nu):
#     return r_j/(p.alpha*nu)**((p.alpha*nu))\
#             *w_j/((1-p.alpha)*nu)**((1-p.alpha)*nu) \
#             *P_mj/(1-nu)**(1-nu)

def unit_cost(r,w,P_m,p,nu):
    return (r/(p.alpha*nu))**(p.alpha*nu) \
        *(w/((1-p.alpha)*nu))**((1-p.alpha)*nu) \
        *(P_m/(1-nu))**(1-nu)

In [6]:
# 使用残差法求解方程，可以用不动点理解，但实际上用官方的包来写
def solve_prices_given_wage(wage,trade_cost,p,cp,max_iter=1000,tol=1e-6):
    P_x=np.ones(cp.N)
    P_m=np.ones(cp.N)
    for i in range(max_iter):
        r=(p.phi_state_1/p.beta+p.phi_state_2)*P_x
        u_m=unit_cost(r,wage,P_m,p,cp.nu_m)

        P_m_new=p.gamma_term*(np.sum((u_m*trade_cost)**(-p.theta)*cp.T_m,axis=1))**(-1/p.theta) #更新
        P_x_mew=unit_cost(r,wage,P_m,p,cp.nu_x)/cp.A_x

        distance=max(
            np.max(np.abs(P_m_new/P_m-1)),
            np.max(np.abs(P_x_mew/P_x-1))
                   )
        P_m=P_m_new
        P_x=P_x_mew
        if distance<tol :
            break
    r=(p.phi_state_1/p.beta+p.phi_state_2)*P_x
    P_c=(1/cp.A_c)*unit_cost(r,wage,P_m,p,cp.nu_c)
    return P_x,P_m,P_c,r

In [7]:
def compute_trade_share(wage,p,cp,trade_cost,r,P_m):
    u_m=unit_cost(r,wage,P_m,p,cp.nu_m)
    numerator=(u_m*trade_cost)**(-p.theta)*cp.T_m
    denominator=np.sum((u_m*trade_cost)**(-p.theta)*cp.T_m,axis=1)
    return numerator / denominator[:, None]
# ij代表了i买j的比例。

In [8]:
### 简单到复杂，整合函数代码，减少内部反复调用和重复计算
@dataclass(frozen=True)
class Quantities:
    K: np.ndarray
    X: np.ndarray
    C: np.ndarray
    Y_m: np.ndarray

    L_c: np.ndarray
    L_x: np.ndarray
    L_m: np.ndarray

    K_c: np.ndarray
    K_x: np.ndarray
    K_m: np.ndarray

    M_c: np.ndarray
    M_x: np.ndarray
    M_m: np.ndarray
    M_use: np.ndarray

    deficit: np.ndarray

def compute_quantity(r,wage,p,cp,P_x,P_c,P_m,trade_share):
    #根据猜测的工资和四组价格，求出来C和KXM及其分配等13个变量
    K=p.alpha/(1-p.alpha)*wage * cp.L / r
    X=K*p.delta
    C=(r*K+wage*cp.L+cp.F-P_x*X)/P_c #根据均衡条件，CP.F也就是净出口为qA

    A1=(1-cp.nu_c)*P_c*C
    A2=(1-cp.nu_x)*P_x*X
    B1=trade_share.T@(A1+A2)
    B2=np.diag(P_m)-trade_share.T@np.diag((1-cp.nu_m)*P_m)
    Y_m=np.linalg.solve(B2,B1)

    L_c=(1-p.alpha)*cp.nu_c*P_c*C/wage
    L_x=(1-p.alpha)*cp.nu_x*P_x*X/wage
    L_m=(1-p.alpha)*cp.nu_m*P_m*Y_m/wage

    K_c=p.alpha/(1-p.alpha)*L_c*wage/r
    K_x=p.alpha/(1-p.alpha)*L_x*wage/r
    K_m=p.alpha/(1-p.alpha)*L_m*wage/r      

    M_c=(1-cp.nu_c)*P_c*C/ P_m
    M_x=(1-cp.nu_x)*P_x*X/ P_m
    M_m=(1-cp.nu_m)*P_m*Y_m/ P_m
    
    M_use=M_c+M_x+M_m

    deficit=P_m*M_use-P_m*Y_m
    return Quantities(
    K=K,
    X=X,
    C=C,
    Y_m=Y_m,
    L_c=L_c,
    L_x=L_x,
    L_m=L_m,
    K_c=K_c,
    K_x=K_x,
    K_m=K_m,
    M_c=M_c,
    M_x=M_x,
    M_m=M_m,
    M_use=M_use,
    deficit=deficit,
)

In [9]:
def wage_residual(log_wage,p,cp,trade_cost):
    wage_raw=np.exp(log_wage)
    wage = wage_raw / np.sum(wage_raw * cp.L)
    P_x, P_m, P_c, r=solve_prices_given_wage(wage,trade_cost,p,cp)
    trade_share=compute_trade_share(wage,p,cp,trade_cost,r,P_m)
    quantity=compute_quantity(r,wage,p,cp,P_x,P_c,P_m,trade_share)
    residual=(cp.F-quantity.deficit)/np.maximum(wage*cp.L,1e-12) #防止除以0 maximum是主元素比较
    return residual

In [10]:
def solve_steady_state(p,cp,trade_cost):
    log_wage0=np.zeros(cp.N)
    fit=least_squares(
        wage_residual,
        x0=log_wage0,
        args=(p,cp,trade_cost),
        ftol=1e-8,
        xtol=1e-8,
        gtol=1e-8,
        max_nfev=50_000,
    )
    if not fit.success:
        raise RuntimeError(f"工资系统求解失败：{fit.message}")
    #这就算完了，然后重新还原一遍
    wage_raw = np.exp(fit.x)
    wage = wage_raw / np.sum(wage_raw * cp.L)

    P_x, P_m, P_c, r = solve_prices_given_wage(
        wage, trade_cost, p, cp
    )

    ts = compute_trade_share(wage, p, cp, trade_cost, r, P_m)

    qnt = compute_quantity(
        r, wage, p, cp, P_x, P_c, P_m, ts
    )

    return {
        "wage": wage,
        "r": r,
        "P_x": P_x,
        "P_m": P_m,
        "P_c": P_c,
        "trade_share": ts,
        "quantity": qnt,
        "residual": wage_residual(fit.x, p, cp, trade_cost),
        "fit": fit,
    }

In [11]:
countries=country["country"].to_list()
observed_trade_share  = (
    trade
    .pivot(index="destination", columns="origin", values="trade_share")
    .reindex(index=countries, columns=countries)
    .to_numpy()
)
trade_costs= (
    trade
    .pivot(index="destination", columns="origin", values="trade_cost")
    .reindex(index=countries, columns=countries)
    .to_numpy()
)

In [12]:
pct_reduce = 0.20
trade_cost_terminal = 1 + (trade_costs - 1) * (1 - pct_reduce)
np.fill_diagonal(trade_cost_terminal, 1.0)

In [13]:
ss_initial = solve_steady_state(p, cp, trade_costs)
ss_terminal= solve_steady_state(p, cp, trade_cost_terminal)

In [14]:
print(np.mean(ss_terminal["wage"]/ss_initial["wage"]-1))
print(np.mean(ss_terminal["quantity"].K/ss_initial["quantity"].K-1))
print(np.mean(ss_terminal["quantity"].C/ss_initial["quantity"].C-1))
print(np.mean((ss_terminal["P_x"]/ss_terminal["P_c"])/(ss_initial["P_x"]/ss_initial["P_c"])-1))

0.11830443799928834
0.20369189776425903
0.15343871453547428
-0.038625299679152375


### 动态路径

上面求解完比较复杂的终端和初始均衡了，也就是会看变化了，但是对于动态模型远远不够，我们要知道过渡动态，先把未知路径设为工资、消费、投资和世界利率。资本和净外国资产不是直接猜测变量，而是由状态方程向前递推。

In [49]:
def solve_prices_given_wage_and_rent(wage,r,trade_cost,p,cp,initial_P_m,max_iter=2000,tol=1e-6):
    P_x=np.ones(cp.N)
    P_m = np.ones(cp.N) if initial_P_m is None else np.asarray(initial_P_m, dtype=float)

    for i in range(max_iter):
        u_m=unit_cost(r,wage,P_m,p,cp.nu_m)
        P_m_new=p.gamma_term*(np.sum((u_m*trade_cost)**(-p.theta)*cp.T_m,axis=1))**(-1/p.theta) #更新
        P_x_new=unit_cost(r,wage,P_m,p,cp.nu_x)/cp.A_x

        distance=max(
            np.max(np.abs(P_m_new/P_m-1)),
            np.max(np.abs(P_x_new/P_x-1))
                   )
        P_m=P_m_new
        P_x=P_x_new
        if distance<tol :
            break
    P_c = unit_cost(r, wage, P_m, p, cp.nu_c) / cp.A_c
    return P_x, P_m, P_c


In [50]:

def phi1_term(K_current,K_next,p):
    ratio =K_next/K_current
    phi1=p.delta**((p.lamb-1)/p.lamb)*(1/p.lamb)*(ratio-1+p.delta)**((1-p.lamb)/p.lamb)
    return phi1

def phi2_term(K_current,K_next,p):
    ratio =K_next/K_current
    phi2=phi1_term(K_current,K_next,p)*((p.lamb-1)*ratio-p.lamb*(1-p.delta))
    return phi2

In [51]:
@dataclass(frozen=True)
class TransitionAllocation:
    """保存给定动态路径后推导出的所有关键变量。"""

    wage: np.ndarray
    q: np.ndarray
    C: np.ndarray
    X: np.ndarray
    K: np.ndarray
    A: np.ndarray
    B: np.ndarray
    r: np.ndarray
    P_x: np.ndarray
    P_m: np.ndarray
    P_c: np.ndarray
    Y_m: np.ndarray
    deficit: np.ndarray
    trade_share: np.ndarray

# 给定猜测的东西，推导所有的内生变量
def allocate_transition(wage_path,q_path,C_path,X_path,K_initial,A_initial,p,cp,trade_cost_path):
    N,T=C_path.shape
    K=np.zeros((N,T+1))
    A=np.zeros((N,T+1))
    r = np.zeros((N, T))
    P_x = np.zeros((N, T))
    P_m = np.zeros((N, T))
    P_c = np.zeros((N, T))
    Y_m = np.zeros((N, T))
    B = np.zeros((N, T))
    deficit = np.zeros((N, T))
    trade_share_path = np.zeros((N, N, T))

    K[:,0]=K_initial
    A[:,0]=A_initial

    previous_P_m=None

    for t in range(T):
        r[:,t]=p.alpha/(1-p.alpha)*(wage_path[:,t]*cp.L)/K[:,t]

        P_x[:,t], P_m[:,t], P_c[:,t]=solve_prices_given_wage_and_rent(wage_path[:,t],r[:,t],trade_cost_path[:, :, t],p,cp,initial_P_m=previous_P_m,)
        previous_P_m=P_m[:,t]

        trade_share_path[:,:,t]=compute_trade_share(wage_path[:,t],p,cp,trade_cost_path[:, :, t],r[:,t],P_m[:,t])

        A1=(1-cp.nu_c)*P_c[:,t]*C_path[:,t]
        A2=(1-cp.nu_x)*P_x[:,t]*X_path[:,t]

        B1=trade_share_path[:,:,t].T@(A1+A2)
        B2=np.diag(P_m[:,t])-trade_share_path[:,:,t].T@np.diag((1-cp.nu_m)*P_m[:,t])
        Y_m[:,t]=np.linalg.solve(B2,B1)
      
        #这里可以继续推算L和K的分配，这里的主要意义在于封闭模型
        L_c = (1 - p.alpha) * cp.nu_c * P_c[:, t] * C_path[:, t] / wage_path[:,t]
        L_x = (1 - p.alpha) * cp.nu_x * P_x[:, t] * X_path[:, t] / wage_path[:,t]
        L_m = (1 - p.alpha) * cp.nu_m * P_m[:, t] * Y_m[:, t] / wage_path[:,t]

        K_c = p.alpha * cp.nu_c * P_c[:, t] * C_path[:, t] / r[:, t]
        K_x = p.alpha * cp.nu_x * P_x[:, t] * X_path[:, t] / r[:, t]
        K_m = p.alpha * cp.nu_m * P_m[:, t] * Y_m[:, t] / r[:, t]

        M_c_t=(1-cp.nu_c)*P_c[:,t]*C_path[:,t]/ P_m[:,t]
        M_x_t=(1-cp.nu_x)*P_x[:,t]*X_path[:,t]/ P_m[:,t]
        M_m_t=(1-cp.nu_m)*P_m[:,t]*Y_m[:,t]/ P_m[:,t]
        
        M_use=M_c_t+M_x_t+M_m_t

        deficit[:,t]=P_m[:,t]*M_use-P_m[:,t]*Y_m[:,t]

        B[:,t]=r[:,t]*K[:,t]+wage_path[:,t]*cp.L+q_path[t]*A[:,t]-P_c[:,t]*C_path[:,t]-P_x[:,t]*X_path[:,t]
        A[:,t+1]=A[:,t]+ B[:,t]
        K[:,t+1]=(1-p.delta)*K[:,t]+p.delta**(1-p.lamb)*X_path[:,t]**p.lamb*K[:,t]**(1-p.lamb)
        
    return TransitionAllocation(
    wage=wage_path,
    q=q_path,
    C=C_path,
    X=X_path,
    K=K,
    A=A,
    B=B,
    r=r,
    P_x=P_x,
    P_m=P_m,
    P_c=P_c,
    Y_m=Y_m,
    deficit=deficit,
    trade_share=trade_share_path,
    )


In [52]:

#这段的作用是为了适应Python支持只支持传入一维向量所编写的中间态
def pack_transition_vars(wage_path,C_path,X_path,q_path):
    return np.r_[
        np.log(wage_path).ravel(),
        np.log(C_path).ravel(),
        np.log(X_path).ravel(),
        q_path.ravel()
    ]

def unpack_transition_vars(flat_vars, cp, T):
    N=cp.N
    block = N * T
    log_wage=flat_vars[0:block].reshape(N,T)
    log_C=flat_vars[block:2*block].reshape(N,T)
    log_X = flat_vars[2 * block:3 * block].reshape(N, T)
    q_path = flat_vars[3 * block:3 * block + T] # T个世界利率 ，3NT+T

    wage_raw=np.exp(log_wage)
    wage = wage_raw / np.sum(wage_raw * cp.L[:, None], axis=0)
    return wage,np.exp(log_C),np.exp(log_X),q_path


In [53]:

def transition_residual(flat_vars, ss_initial, ss_terminal, cp, p, trade_cost_path, T):
    #首先，我们解包路径,传回来有价值的经济变量
    wage_path,C_path,X_path,q_path=unpack_transition_vars(flat_vars,cp,T)
    K_initial=ss_initial["quantity"].K
    A_initial=cp.F/p.q

    TA= allocate_transition(wage_path,q_path,C_path,X_path,K_initial,A_initial,p,cp,trade_cost_path)
    K_terminal=ss_terminal["quantity"].K
    P_c_terminal=ss_terminal["P_c"]
    P_x_terminal=ss_terminal["P_x"]
    P_m_terminal=ss_terminal["P_m"]
    r_terminal=ss_terminal["r"]
    C_terminal=ss_terminal["quantity"].C
    q_terminal=p.q

    res_euler_c = np.zeros((cp.N, T))
    res_euler_k = np.zeros((cp.N, T))
    res_market = np.zeros((cp.N, T))
    res_global_assets = np.zeros(T)

    for t in range(T):
        #这里要处理关于边界的问题
        C_t = C_path[:, t]
        P_c_t = TA.P_c[:, t]
        P_x_t = TA.P_x[:, t]
        if t<T-1:#正常来说直接提取就OK
            C_next = C_path[:, t + 1]
            P_c_next = TA.P_c[:, t + 1]
            P_x_next = TA.P_x[:, t + 1]
            r_next = TA.r[:, t + 1]
            q_next = q_path[t + 1]
            K_next = TA.K[:, t + 1]
            K_next2 = TA.K[:, t + 2]
        else:#到边界了注意资本处理
            C_next = C_terminal
            P_c_next = P_c_terminal
            P_x_next = P_x_terminal
            r_next = r_terminal
            q_next = q_terminal
            K_next = K_terminal
            K_next2 = K_terminal
        
        eule_c=C_next/C_t-((p.beta*(1+q_next))/(P_c_next/P_c_t))**p.sigma
        res_euler_c[:,t]=eule_c

        K_t=TA.K[:,t]
        ph1=phi1_term(K_t,K_next,p)
        ph2=phi2_term(K_next,K_next2,p)
        eule_k=C_next/C_t-(p.beta*((r_next/P_x_next-ph2)/ph1)*((P_x_next/P_c_next)/(P_x_t/P_c_t)))**p.sigma
        res_euler_k[:,t]=eule_k

        mkt_balance = TA.deficit[:, t] + TA.B[:, t] - q_path[t] * TA.A[:, t]
        res_market[:, t] = mkt_balance / np.maximum(wage_path[:, t] * cp.L, 1e-12)

        res_global_assets[t] = np.sum(TA.B[:, t]) / np.sum(wage_path[:, t] * cp.L)

    return np.r_[
        res_euler_c.ravel(),       # 大小为 N * T
        res_euler_k.ravel(),       # 大小为 N * T
        res_market.ravel(),        # 大小为 N * T
        res_global_assets.ravel()  # 大小为 T
    ]


In [54]:
T = 150  # 设定过渡期长度为 150 期
N = cp.N

trade_cost_path = np.zeros((N, N, T))
for t in range(T):
    trade_cost_path[:, :, t] = trade_cost_terminal #瞬间成本升高

# 2. 构造初始猜测路径 x0 (线性插值)

wage_init = np.zeros((N, T))
C_init = np.zeros((N, T))
X_init = np.zeros((N, T))
q_init = np.ones(T) * p.q  # 利率初值猜为稳态利率

for i in range(N):
    # 消费和投资在初值和终值之间平滑拉一条直线
    wage_init[i, :] = np.linspace(ss_initial["wage"][i], ss_terminal["wage"][i], T)
    C_init[i, :] = np.linspace(ss_initial["quantity"].C[i], ss_terminal["quantity"].C[i], T)
    X_init[i, :] = np.linspace(ss_initial["quantity"].X[i], ss_terminal["quantity"].X[i], T)

# 打包成 1D 向量
x0 = pack_transition_vars(wage_init, C_init, X_init, q_init)


In [ ]:

# res = least_squares(
#     transition_residual,
#     x0=x0,
#     args=(ss_initial, ss_terminal, cp, p, trade_cost_path, T),
#     method='trf',                  # 信赖域算法
#     ftol=1e-6,
#     xtol=1e-6,
#     gtol=1e-6,
#     max_nfev=1500,
#     verbose=2                      # 打印迭代过程以便监控收敛
# )
# #求解完毕，拆包
# w_opt, C_opt, X_opt, q_opt = unpack_transition_vars(res.x, cp, T)
    

# K_initial = ss_initial["quantity"].K
# A_initial = cp.F / p.q
# sim_opt = allocate_transition(
#     w_opt, q_opt, C_opt, X_opt, K_initial, A_initial, p, cp, trade_cost_path
# )

In [153]:
fun = res.fun
print("success:", res.success)
print("message:", res.message)
print("cost:", res.cost)
print("max abs residual:", np.max(np.abs(fun)))
print("rms residual:", np.sqrt(np.mean(fun**2)))

M = cp.N * T
print("Euler C max:", np.max(np.abs(fun[0:M])))
print("Euler K max:", np.max(np.abs(fun[M:2*M])))
print("Market max:", np.max(np.abs(fun[2*M:3*M])))
print("Global asset max:", np.max(np.abs(fun[3*M:3*M+T])))

success: True
message: `ftol` termination condition is satisfied.
cost: 0.008305723204736099
max abs residual: 0.04821901624715641
rms residual: 0.0035340924337699804
Euler C max: 0.04821901624715641
Euler K max: 0.030337930752046938
Market max: 0.01540112440096839
Global asset max: 0.008212929866063028


### 工程化

本质上，上面属于头铁复刻了作者在原文批判的直接求解，确实巨慢无比，但是我觉得这还有很多的优化空间，没必要直接换个算法，我注意到求解大型矩阵的时候，稀疏矩阵由特别的存储方式和计算方法。我人为的设定了矩阵的稀疏模式，发现效果顿时就差强人意了。虽然似乎还有很多花里胡哨的算法，但是稍微的优化可以带来更高的精度和更快的速度以及更简洁易读的代码。
当然其实这离着工程化十万八千里，虽然本质是只是添加了稀疏矩阵，但把中间的嵌套不动点删除了，直接放进优化里面，因此重写了完整代码

In [68]:
# 辅助函数，资产边际调整成本
def phi1_term(K_current,K_next,p):
    ratio =K_next/K_current
    phi1=p.delta**((p.lamb-1)/p.lamb)*(1/p.lamb)*(ratio-1+p.delta)**((1-p.lamb)/p.lamb)
    return phi1

def phi2_term(K_current,K_next,p):
    ratio =K_next/K_current
    phi2=phi1_term(K_current,K_next,p)*((p.lamb-1)*ratio-p.lamb*(1-p.delta))
    return phi2

In [ ]:
def get_transition_sparsity(N, T):
    """
    生成系统的稀疏结构。
    每个时期的方程组仅依赖于 t-1, t, t+1 期的变量。
    """
    D = 6 * N + 1  # 每个时期的变量数: w, C, X, Pm, K_next, A_next, q
    M = D * T      # 总变量数和方程数
    rows = []
    cols = []
    
    for t in range(T):
        row_start = t * D
        row_end = (t + 1) * D
        
        # 依赖的时间步：t-1, t, t+1
        neighbors = [t]
        if t > 0:
            neighbors.append(t - 1)
        if t < T - 1:
            neighbors.append(t + 1)
            
        for t_col in neighbors:
            col_start = t_col * D
            col_end = (t_col + 1) * D
            
            # 将这些块设为潜在非零区域（密集子块）
            r, c = np.meshgrid(np.arange(row_start, row_end), 
                               np.arange(col_start, col_end), 
                               indexing='ij')
            rows.extend(r.ravel())
            cols.extend(c.ravel())
            
    return sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(M, M))


# =============================================================================
# 2. 变量的打包与解包（一维向量 <-> 经济变量）
# =============================================================================
def pack_transition_vars(wage, C, X, Pm, K_next, A_next, q, N, T):
    """
    将高维经济变量打包为 1D 向量
    为了确保数值稳定性，对严格正值变量取 log 
    """
    V = np.zeros((T, 6 * N + 1))
    V[:, 0:N] = np.log(wage).T
    V[:, N:2*N] = np.log(C).T
    V[:, 2*N:3*N] = np.log(X).T
    V[:, 3*N:4*N] = np.log(Pm).T
    V[:, 4*N:5*N] = np.log(K_next).T # K_1 到 K_T
    V[:, 5*N:6*N] = A_next.T         # 净国外资产可以为负，保持原值
    V[:, 6*N] = q
    return V.ravel()

def unpack_transition_vars(V, N, T, cp):
    """从 1D 向量解包"""
    V_reshaped = V.reshape(T, 6 * N + 1)
    
    # 提取工资并应用你原先的劳动力计价基准标准化
    wage_raw = np.exp(V_reshaped[:, 0:N].T)
    wage = wage_raw / np.sum(wage_raw * cp.L[:, None], axis=0)
    
    C = np.exp(V_reshaped[:, N:2*N].T)
    X = np.exp(V_reshaped[:, 2*N:3*N].T)
    Pm = np.exp(V_reshaped[:, 3*N:4*N].T)
    K_endog = np.exp(V_reshaped[:, 4*N:5*N].T) # K_1 到 K_T
    A_endog = V_reshaped[:, 5*N:6*N].T         # A_1 到 A_T
    q = V_reshaped[:, 6*N]
    
    return wage, C, X, Pm, K_endog, A_endog, q


# =============================================================================
# 3. 平坦化（无内层循环）残差计算函数
# =============================================================================
def transition_residual_flat(V, ss_initial, ss_terminal, cp, p, trade_cost_path, T):
    N = cp.N
    # 1. 解包猜测变量
    wage, C, X, Pm, K_endog, A_endog, q = unpack_transition_vars(V, N, T, cp)
    
    K_initial = ss_initial["quantity"].K
    A_initial = cp.F / p.q
    
    # 2. 重构包含 t=0 初始状态的完整状态序列
    K = np.zeros((N, T + 1))
    K[:, 0] = K_initial
    K[:, 1:] = K_endog  # K[:, t] 代表第 t 期的资本
    
    A = np.zeros((N, T + 1))
    A[:, 0] = A_initial
    A[:, 1:] = A_endog
    
    # 3. 准备残差容器
    res_price = np.zeros((N, T))
    res_euler_c = np.zeros((N, T))
    res_euler_k = np.zeros((N, T))
    res_k = np.zeros((N, T))
    res_a = np.zeros((N, T))
    res_mkt = np.zeros((N, T))
    res_global = np.zeros(T)
    
    # 预留用于跨期计算的价格与流量容器
    P_c = np.zeros((N, T))
    P_x = np.zeros((N, T))
    r = np.zeros((N, T))
    B = np.zeros((N, T))
    deficit = np.zeros((N, T))
    
    # --- 第一遍循环：计算各期内的静态贸易、价格平衡和流量 ---
    for t in range(T):
        w_t = wage[:, t]
        Pm_t = Pm[:, t]
        K_t = K[:, t]
        q_t = q[t]
        A_t = A[:, t]
        C_t = C[:, t]
        X_t = X[:, t]
        trade_cost_t = trade_cost_path[:, :, t]
        
        # 计算利率 r
        r[:, t] = p.alpha / (1 - p.alpha) * (w_t * cp.L) / K_t
        r_t = r[:, t]
        
        # 关键的一步：无须循环！直接利用 Pm_t 计算理论单位成本和预估价格
        u_m = unit_cost(r_t, w_t, Pm_t, p, cp.nu_m)
        Pm_calc = p.gamma_term * (np.sum((u_m * trade_cost_t)**(-p.theta) * cp.T_m, axis=1))**(-1/p.theta)
        
        # 进口价格残差方程：迫使猜测的 Pm_t 等于根据定价公式推导出的 Pm_calc
        res_price[:, t] = np.log(Pm_t) - np.log(Pm_calc)
        
        # 导出其他价格
        P_x[:, t] = unit_cost(r_t, w_t, Pm_t, p, cp.nu_x) / cp.A_x
        P_c[:, t] = unit_cost(r_t, w_t, Pm_t, p, cp.nu_c) / cp.A_c
        
        # 贸易份额及产品流动求解
        trade_share_t = compute_trade_share(w_t, p, cp, trade_cost_t, r_t, Pm_t)
        
        A1 = (1 - cp.nu_c) * P_c[:, t] * C_t
        A2 = (1 - cp.nu_x) * P_x[:, t] * X_t
        
        B1 = trade_share_t.T @ (A1 + A2)
        B2 = np.diag(Pm_t) - trade_share_t.T @ np.diag((1 - cp.nu_m) * Pm_t)
        Y_m_t = np.linalg.solve(B2, B1)
        
        M_c_t = (1 - cp.nu_c) * P_c[:, t] * C_t / Pm_t
        M_x_t = (1 - cp.nu_x) * P_x[:, t] * X_t / Pm_t
        M_m_t = (1 - cp.nu_m) * Pm_t * Y_m_t / Pm_t
        M_use = M_c_t + M_x_t + M_m_t
        
        deficit[:, t] = Pm_t * M_use - Pm_t * Y_m_t
        
        # 净国外金融流出 B_t
        B[:, t] = r_t * K_t + w_t * cp.L + q_t * A_t - P_c[:, t] * C_t - P_x[:, t] * X_t
        
        # 资本积累残差方程
        K_next_t_calc = (1 - p.delta) * K_t + p.delta**(1 - p.lamb) * X_t**p.lamb * K_t**(1 - p.lamb)
        res_k[:, t] = np.log(K[:, t+1]) - np.log(K_next_t_calc)
        
        # 资产积累残差方程
        res_a[:, t] = A[:, t+1] - (A_t + B[:, t])
        
        # 国家层面国际收支平衡残差
        mkt_balance = deficit[:, t] + B[:, t] - q_t * A_t
        res_mkt[:, t] = mkt_balance / np.maximum(w_t * cp.L, 1e-12)
        
        # 全球总资产平衡残差
        res_global[t] = np.sum(B[:, t]) / np.sum(w_t * cp.L)

    # --- 第二遍循环：处理跨期欧拉方程残差 ---
    K_terminal = ss_terminal["quantity"].K
    P_c_terminal = ss_terminal["P_c"]
    P_x_terminal = ss_terminal["P_x"]
    r_terminal = ss_terminal["r"]
    C_terminal = ss_terminal["quantity"].C
    q_terminal = p.q
    
    for t in range(T):
        C_t = C[:, t]
        P_c_t = P_c[:, t]
        P_x_t = P_x[:, t]
        
        if t < T - 1:
            C_next = C[:, t + 1]
            P_c_next = P_c[:, t + 1]
            P_x_next = P_x[:, t + 1]
            r_next = r[:, t + 1]
            q_next = q[t + 1]
            K_next = K[:, t + 1]
            K_next2 = K[:, t + 2] if t < T - 2 else K_terminal
        else:
            C_next = C_terminal
            P_c_next = P_c_terminal
            P_x_next = P_x_terminal
            r_next = r_terminal
            q_next = q_terminal
            K_next = K_terminal
            K_next2 = K_terminal
            
        # 消费欧拉残差
        euler_c = C_next / C_t - ((p.beta * (1 + q_next)) / (P_c_next / P_c_t))**p.sigma
        res_euler_c[:, t] = euler_c
        
        # 资本欧拉残差
        K_t = K[:, t]
        ph1 = phi1_term(K_t, K_next, p)
        ph2 = phi2_term(K_next, K_next2, p)
        euler_k = C_next / C_t - (p.beta * ((r_next / P_x_next - ph2) / ph1) * ((P_x_next / P_c_next) / (P_x_t / P_c_t)))**p.sigma
        res_euler_k[:, t] = euler_k

    # --- 将所有残差按时间打包输出 ---
    res_flat = np.zeros((T, 6 * N + 1))
    res_flat[:, 0:N] = res_price.T
    res_flat[:, N:2*N] = res_euler_c.T
    res_flat[:, 2*N:3*N] = res_euler_k.T
    res_flat[:, 3*N:4*N] = res_k.T
    res_flat[:, 4*N:5*N] = res_a.T
    res_flat[:, 5*N:6*N] = res_mkt.T
    res_flat[:, 6*N] = res_global
    
    return res_flat.ravel()


# 4. 执行求解
T = 150  # 设定过渡期长度为 150 期
N = cp.N

trade_cost_path = np.zeros((N, N, T))
for t in range(T):
    trade_cost_path[:, :, t] = trade_cost_terminal #瞬间成本升高

wage_init = np.zeros((N, T))
C_init = np.zeros((N, T))
X_init = np.zeros((N, T))
Pm_init = np.zeros((N, T))
K_init = np.zeros((N, T))  # K_1 到 K_T
A_init = np.zeros((N, T))  # A_1 到 A_T
q_init = np.ones(T) * p.q

Pm_initial = ss_initial["P_m"]
Pm_terminal = ss_terminal["P_m"]
K_initial = ss_initial["quantity"].K
K_terminal = ss_terminal["quantity"].K
A_initial = cp.F / p.q
A_terminal = ss_terminal.get("A", A_initial)

for i in range(N):
    wage_init[i, :] = np.linspace(ss_initial["wage"][i], ss_terminal["wage"][i], T)
    C_init[i, :] = np.linspace(ss_initial["quantity"].C[i], ss_terminal["quantity"].C[i], T)
    X_init[i, :] = np.linspace(ss_initial["quantity"].X[i], ss_terminal["quantity"].X[i], T)
    Pm_init[i, :] = np.linspace(Pm_initial[i], Pm_terminal[i], T)
    
    K_path_full = np.linspace(K_initial[i], K_terminal[i], T + 1)
    K_init[i, :] = K_path_full[1:] # 截取 1 到 T 期的资本
    
    A_path_full = np.linspace(A_initial[i], A_terminal[i], T + 1)
    A_init[i, :] = A_path_full[1:] # 截取 1 到 T 期的资产

V0 = pack_transition_vars(wage_init, C_init, X_init, Pm_init, K_init, A_init, q_init, N, T)

sparsity_pattern = get_transition_sparsity(N, T)

res = least_squares(
    transition_residual_flat,
    x0=V0,
    jac_sparsity=sparsity_pattern,  # 稀疏
    args=(ss_initial, ss_terminal, cp, p, trade_cost_path, T),
    method='trf',                  
    ftol=1e-3,                      # 
    gtol=1e-5,                      # 
    xtol=1e-4,                      # 
    max_nfev=1500,
    verbose=2                      
)
w_opt, C_opt, X_opt, Pm_opt, K_opt, A_opt, q_opt = unpack_transition_vars(res.x, cp.N, T, cp)

   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         3.2626e+01                                    3.18e+00    
       1              2         1.8787e-01      3.24e+01       1.10e+01       2.60e-01    
       2              3         1.0942e-02      1.77e-01       5.04e+00       7.37e-02    
       3              4         6.8519e-06      1.09e-02       7.55e-02       2.59e-04    
`xtol` termination condition is satisfied.
Function evaluations 4, initial cost 3.2626e+01, final cost 6.8519e-06, first-order optimality 2.59e-04.


In [59]:
def split_transition_residual(flat_residual, N, T):
    R = flat_residual.reshape(T, 6 * N + 1)
    return {
        "price": R[:, 0:N].T,
        "euler_c": R[:, N:2*N].T,
        "euler_k": R[:, 2*N:3*N].T,
        "capital_law": R[:, 3*N:4*N].T,
        "asset_law": R[:, 4*N:5*N].T,
        "market_clear": R[:, 5*N:6*N].T,
        "global_asset": R[:, 6*N],
    }


def report_residual_blocks(fit, N, T, countries=None):
    flat = transition_residual_flat(
        fit.x,
        ss_initial,
        ss_terminal,
        cp,
        p,
        trade_cost_path,
        T,
    )

    print("overall")
    print("  success:", fit.success)
    print("  message:", fit.message)
    print("  cost:", fit.cost)
    print("  l2:", np.linalg.norm(flat))
    print("  rms:", np.sqrt(np.mean(flat ** 2)))
    print("  max abs:", np.max(np.abs(flat)))
    print("  n residual:", flat.size)

    blocks = split_transition_residual(flat, N, T)

    for name, block in blocks.items():
        abs_block = np.abs(block)
        print()
        print(name)
        print("  max abs:", np.max(abs_block))
        print("  rms:", np.sqrt(np.mean(block ** 2)))
        print("  l2:", np.linalg.norm(block))

        if block.ndim == 2:
            i, t = np.unravel_index(np.argmax(abs_block), block.shape)
            country = countries[i] if countries is not None else i
            print("  max country:", country)
            print("  max time:", t)
            print("  signed value:", block[i, t])
        else:
            t = int(np.argmax(abs_block))
            print("  max time:", t)
            print("  signed value:", block[t])
countries = country["country"].tolist()
report_residual_blocks(res, N, T, countries)

overall
  success: True
  message: `gtol` termination condition is satisfied.
  cost: 6.712737245418331e-06
  l2: 0.0036640789416764294
  rms: 1.8377915712425006e-05
  max abs: 0.00026186088435079316
  n residual: 39750

price
  max abs: 1.3551964142877182e-05
  rms: 1.4754605635238574e-06
  l2: 0.00011986698282593635
  max country: CHN
  max time: 0
  signed value: 1.3551964142877182e-05

euler_c
  max abs: 0.0001480701329504841
  rms: 3.411044661604529e-05
  l2: 0.0027711457830803675
  max country: USA
  max time: 89
  signed value: -0.0001480701329504841

euler_k
  max abs: 8.474623919008106e-05
  rms: 7.624666246049558e-06
  l2: 0.000619430814054381
  max country: NLD
  max time: 149
  signed value: 8.474623919008106e-05

capital_law
  max abs: 2.7246956630477825e-05
  rms: 2.8329769118937883e-06
  l2: 0.00023015213231672123
  max country: USA
  max time: 0
  signed value: -2.7246956630477825e-05

asset_law
  max abs: 0.00026186088435079316
  rms: 2.829087698542212e-05
  l2: 0.0022

In [63]:
w_opt_exact=pd.read_csv(r"C:\Users\27968\Downloads\w_opt.csv")

In [64]:
def split_transition_residual(flat_residual, N, T):
    R = flat_residual.reshape(T, 6 * N + 1)
    return {
        "price": R[:, 0:N].T,
        "euler_c": R[:, N:2*N].T,
        "euler_k": R[:, 2*N:3*N].T,
        "capital_law": R[:, 3*N:4*N].T,
        "asset_law": R[:, 4*N:5*N].T,
        "market_clear": R[:, 5*N:6*N].T,
        "global_asset": R[:, 6*N],
    }


def report_residual_blocks(fit, N, T, countries=None):
    flat = transition_residual_flat(
        fit.x,
        ss_initial,
        ss_terminal,
        cp,
        p,
        trade_cost_path,
        T,
    )

    print("overall")
    print("  success:", fit.success)
    print("  message:", fit.message)
    print("  cost:", fit.cost)
    print("  l2:", np.linalg.norm(flat))
    print("  rms:", np.sqrt(np.mean(flat ** 2)))
    print("  max abs:", np.max(np.abs(flat)))
    print("  n residual:", flat.size)

    blocks = split_transition_residual(flat, N, T)

    for name, block in blocks.items():
        abs_block = np.abs(block)
        print()
        print(name)
        print("  max abs:", np.max(abs_block))
        print("  rms:", np.sqrt(np.mean(block ** 2)))
        print("  l2:", np.linalg.norm(block))

        if block.ndim == 2:
            i, t = np.unravel_index(np.argmax(abs_block), block.shape)
            country = countries[i] if countries is not None else i
            print("  max country:", country)
            print("  max time:", t)
            print("  signed value:", block[i, t])
        else:
            t = int(np.argmax(abs_block))
            print("  max time:", t)
            print("  signed value:", block[t])
countries = country["country"].tolist()
report_residual_blocks(res, N, T, countries)

overall
  success: True
  message: `xtol` termination condition is satisfied.
  cost: 6.851915195914186e-06
  l2: 0.0037018685000724124
  rms: 1.8567456748539632e-05
  max abs: 0.0002630212635540488
  n residual: 39750

price
  max abs: 1.3886405671570401e-05
  rms: 1.5188783779117333e-06
  l2: 0.00012339426274126094
  max country: CHN
  max time: 0
  signed value: 1.3886405671570401e-05

euler_c
  max abs: 0.0001522433430506398
  rms: 3.4295459501709404e-05
  l2: 0.002786176300965244
  max country: USA
  max time: 93
  signed value: -0.0001522433430506398

euler_k
  max abs: 8.475426641407768e-05
  rms: 7.811151506854773e-06
  l2: 0.0006345809482611823
  max country: NLD
  max time: 149
  signed value: 8.475426641407768e-05

capital_law
  max abs: 2.7129201098041733e-05
  rms: 2.9154848589020086e-06
  l2: 0.00023685510961854567
  max country: USA
  max time: 0
  signed value: -2.7129201098041733e-05

asset_law
  max abs: 0.0002630212635540488
  rms: 2.827849427700032e-05
  l2: 0.00229

In [67]:
corr = np.corrcoef(w_opt, w_opt_exact)[0,1]
print(corr)

0.9910755076877138
